# A9 · Opening up the model, part 2: is it fair to every group?

**Where we are**

```
A8     the final model: 86% on time at P75 on Jul–Aug (today's 15 min: 70%)
A9     part 1 ✅ why it says what it says (pushes)
       HERE, part 2: does that 86% hold for EVERY group, or is someone quietly worse off?
```

**The one idea:** an average can hide a group. "86% on time" could be 95% for some patients and 60% for others. So we cut the results by groups that matter and ask each group two questions:

1. **Is it clearly below the average, beyond luck?** A small group wobbles by luck alone (A5's 1/√n idea). The **luck band** says how much.
2. **Is it better or worse off than today?** Today's rule books 15 minutes for everyone.

Then, for every group that's flagged: **where does the gap come from?** Three possible sources:

| Source | How to recognise it |
|---|---|
| **Drift** (the world changed in July) | the gap is in July–August but **not in June** |
| **The booking grid** (rounding up to 5 min) | the model covers the group fine **before** rounding; the gap appears **after** |
| **The model itself** | the group is short **even before** rounding, **in June too** |

We use P75, the PO's provisional default. **How big a gap is acceptable is not ours to decide**: that's a business and clinical call for the PO (like q itself, A1). Our job is to measure it honestly.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import pandas as pd

from slotpred import fairness as fa
from slotpred.config import ROOT

Q = 0.75
july = fa.booked_test(Q)     # Jul-Aug, final model (Jan-Jun) + new-doctor fallback: one row per visit
june = fa.booked_june(Q)     # June, the A6 model (Jan-May): before the July changes
print(f"on time  July-Aug {july.on_time.mean():.1%}   June {june.on_time.mean():.1%}   today's 15 min (Jul-Aug) {july.today_on_time.mean():.1%}")

## 1 · The luck band and the group table

**TODO(you):** in [slotpred/fairness.py](../slotpred/fairness.py), write

- `luck_band(p, n)`: two standard deviations of a share, `2 × √(p(1 − p) / n)`. One line.
- `group_table(visits, group)`: one row per group with n, on-time share, today's share, late minutes, gap from the overall share, the luck band and a flag. A `groupby(...).agg(...)` plus three new columns. The docstring lists every column.

Then run the tests.

In [ ]:
!"{sys.executable}" -m pytest "{ROOT / 'tests' / 'test_fairness.py'}" -q

## 2 · Every group, July–August

A row with `flag = True` is below the average by more than luck can explain. Compare `on_time` with `today` too.

In [ ]:
for g in fa.GROUPS:
    display(fa.group_table(july, g).round(3))

## 3 · Where does each gap come from?

For each flagged group: on-time in July–August and in June, and in June **before rounding** (`covered`: the visit fits the model's P75 itself). A well-calibrated P75 covers about 75% of every group before rounding.

In [ ]:
def origin(g):
    jul = fa.group_table(july, g)[["n", "on_time", "flag"]].add_prefix("jul_")
    jun = june.groupby(g).agg(jun_on_time=("on_time", "mean"), jun_before_rounding=("covered", "mean"),
                              jun_avg_slot=("slot", "mean"))
    return jul.join(jun).round(3)

for g in ["age_band", "visit_type", "clinic", "department", "new_doctor"]:
    display(origin(g))

**Questions** (short answers, your own words)
1. **Dermatology** is on time less often than with today's 15 minutes (about 70% vs 81%), and below even the 75% the P75 promises. In June it was 93%. Where does the gap come from? What would you tell the PO, and what are two ways to fix it?
2. **New visits** are on time less often than follow-ups (about 83% vs 88% in July). But look at June *before rounding*: both are covered about 75%. Is the model unfair to new visits? Where does the gap come from? (Hint: your own A2 "In my words" about rounding.)
3. **Babies 0–2** (461 visits): 79% on time vs 86% overall. Is that more than luck? Use the luck band. Then: who is the one doctor who sees only children and is new in July? How would you check whether he explains the babies' gap?
4. **65+** is flagged in July, is lower in June too, and is under 75% even before rounding. Which source is it? Name one fix, and say who decides whether it's needed.
5. **Follow-ups** are also on time less often than today (about 88% vs 92%). Is that a problem like Dermatology's? Hint: compare the slot they get (about 13 minutes instead of 15) and the 75% promise.


*Answers:*